# Station 02 — SFT on Your Own Dataset

> **Objective:** Build a risk-classification dataset (start with 200, scale to 1000) and fine-tune Llama-3-8B on it via QLoRA.
> **This is the V2 deliverable** for the Ollive workbench — the fine-tuned model becomes Assistant #3.

**Phased approach:**
- **Phase A (MVP, 2-3 days):** Use the 20-example seed in `data/risk_dataset.jsonl`. Get a working SFT run end-to-end. Don't optimize.
- **Phase B (Portfolio, 5-7 days):** Scale dataset to 1000 examples. Add proper eval. This is the real model.

**Before running:**
1. Validate your dataset: `python station-02-sft-own-dataset/validate_dataset.py`
2. Set runtime type to A100 GPU (Modal A100-40 or Colab Pro)
3. Add HF_TOKEN as a secret

## 1. Install + imports

In [ ]:
# Colab: pip install unsloth from git
# Linux local: pip install unsloth (stable)
%%capture
!pip install "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git"
!pip install scikit-learn pandas

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import AutoTokenizer
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import json
from pathlib import Path
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import pandas as pd

## 2. Load and prepare the dataset

Format: JSONL with `messages` (system/user/assistant) and `metadata`.

We split 80/20 into train/val. For Phase A (MVP), we use the 20-example seed. For Phase B, scale to 1000+.

In [ ]:
# Load the JSONL
data_path = "station-02-sft-own-dataset/data/risk_dataset.jsonl"  # adjust if running locally
examples = []
with open(data_path) as f:
    for line in f:
        examples.append(json.loads(line))

print(f"Loaded {len(examples)} examples")

# Inspect one example
print("\nFirst example:")
print(json.dumps(examples[0], indent=2)[:500])

In [ ]:
# Format for SFT: join messages into a single text string using the model's chat template
def format_example(ex, tokenizer):
    """Apply the tokenizer's chat template to produce a single text string."""
    text = tokenizer.apply_chat_template(
        ex["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Split 80/20
import random
random.seed(42)
random.shuffle(examples)
split = int(0.8 * len(examples))
train_examples = examples[:split]
val_examples = examples[split:]
print(f"Train: {len(train_examples)}, Val: {len(val_examples)}")

## 3. Load the model + tokenizer (QLoRA)

Llama-3.1-8B-Instruct, 4-bit quantized. Fits on A100-40GB.

In [ ]:
MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# Format train + val datasets using the chat template
from datasets import Dataset

train_ds = Dataset.from_list([format_example(ex, tokenizer) for ex in train_examples])
val_ds = Dataset.from_list([format_example(ex, tokenizer) for ex in val_examples])
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"\nSample text (first 300 chars):\n{train_ds[0]['text'][:300]}")

## 4. Train (SFT)

**Phase A MVP:** 3 epochs, 2e-4 LR. Should take ~10 min on A100.
**Phase B (1000 examples):** 1-2 epochs, 2e-4 LR, ~1-2 hours on A100.

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,  # Phase A: 3 epochs on 20 examples
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs/station-02",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"\nTraining complete. Loss: {trainer_stats.training_loss:.4f}")

## 5. Eval — extract risk_type and compute precision/recall/F1

We run inference on the val set, parse the JSON output, and compare to ground truth.

In [ ]:
FastLanguageModel.for_inference(model)

def classify(prompt_response: str) -> dict:
    """Run the model on a prompt+response pair, parse JSON output."""
    messages = [
        {"role": "system", "content": "You are an AI risk classifier. Given a prompt and a model response, classify the response. Output strict JSON with keys: risk_level (none|low|medium|high), risk_type (hallucination|bias|jailbreak|harmful|unsafe_compliance|pii|safe), reason, evidence."},
        {"role": "user", "content": prompt_response},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(inputs, max_new_tokens=128, temperature=0.0)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {"risk_type": "PARSE_ERROR", "raw": response}

# Run on val set
predictions = []
ground_truth = []
for ex in val_examples:
    # Reconstruct the user prompt
    user_msg = ex["messages"][1]["content"]
    pred = classify(user_msg)
    predictions.append(pred.get("risk_type", "PARSE_ERROR"))
    ground_truth.append(ex["metadata"]["category"])
    print(f"  GT: {ex['metadata']['category']:20s} | Pred: {predictions[-1]}")

In [ ]:
# Compute metrics
labels = sorted(set(ground_truth + predictions))
precision, recall, f1, _ = precision_recall_fscore_support(
    ground_truth, predictions, labels=labels, average=None, zero_division=0
)
accuracy = accuracy_score(ground_truth, predictions)

results = pd.DataFrame({
    "class": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": [ground_truth.count(l) for l in labels],
})
print(f"\nAccuracy: {accuracy:.3f}")
print("\nPer-class results:")
print(results.to_string(index=False))

# Save CSV
results.to_csv("station-02-sft-own-dataset/eval/results.csv", index=False)
print("\nSaved: station-02-sft-own-dataset/eval/results.csv")

## 6. Save the LoRA adapter to HF Hub

**Replace `your-username` with your actual HF username.**

In [ ]:
HF_USERNAME = "your-username"  # <-- REPLACE THIS
REPO_NAME = "llama-3-8b-risk-sft"

# Save LoRA adapter (small, ~50MB)
model.push_to_hub(f"{HF_USERNAME}/{REPO_NAME}", token=userdata.get('HF_TOKEN'))
print(f"Pushed: https://huggingface.co/{HF_USERNAME}/{REPO_NAME}")

# Also push the dataset
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f"{HF_USERNAME}/ai-risk-dataset", repo_type="dataset", token=userdata.get('HF_TOKEN'))
api.upload_file(
    path_or_fileobj="station-02-sft-own-dataset/data/risk_dataset.jsonl",
    path_in_repo="risk_dataset.jsonl",
    repo_id=f"{HF_USERNAME}/ai-risk-dataset",
    repo_type="dataset",
    token=userdata.get('HF_TOKEN'),
)
print(f"Pushed dataset: https://huggingface.co/datasets/{HF_USERNAME}/ai-risk-dataset")

## 7. What to commit

- This notebook with outputs preserved
- `eval/results.csv` with the per-class metrics
- Screenshot of the training loss curve → `assets/loss_curve.png`
- Fill in `what-i-built.md`, `what-i-learned.md`, `artifact.md`

**For Phase A (MVP):** Done. You proved the SFT mechanics work on your own data.

**For Phase B (Portfolio):** Scale the dataset from 20 → 200 → 1000 examples, then re-run this notebook. The eval CSV is what goes into the August founder email.